In [2]:
import time
import os
import re
import shutil
import tempfile
import subprocess
from PIL import Image
from glob import glob
from tqdm import tqdm

FRAME_RE = re.compile(r'^([^_]+_[^_]+)_(\d+)\.png$')       # extract frame ID in each name.

In [5]:
def extract_info(fname):
    m = FRAME_RE.match(fname)
    if m:
        return int(m.group(1)), int(m.group(2))     # video ID, frame ID
    else:
        return None, None
    

def sort_frames(frame_dir):
    frames = [f for f in os.listdir(frame_dir) if f.lower().endswith('png')]
    if not frames:
        raise FileNotFoundError(f'Do not find any `png` frames.')
    
    groups = {}
    for f in frames:
        vid, fid = extract_info(f)
        if vid is None:
            continue
        groups.setdefault(vid, []).append((fid, os.path.join(frame_dir, f)))
        
    sorted_groups = {}
    for vid, items in groups.items():
        items.sort(key=lambda x: x[0])
        sorted_groups[vid] = [path for _, path in items]
        
    return sorted_groups
        


def probe_frame_size(frame_path):
    with Image.open(frame_path) as f:
        return f.width, f.height
    

FPS  = 30
ROOT = '/data/ssd/zhaoy/datasets/TouchandGoDataset-v2/dataset-comp'
DEBUG = False
for mode in tqdm(['train', 'test']):
    frame_dir = os.path.join(ROOT, mode, 'image')
    video_dir = os.path.join(ROOT, mode, 'video', 'yuv')
    os.makedirs(video_dir, exist_ok=True)
    
    groups = sort_frames(frame_dir)
    for vid, frames in groups.items():
        width, height = probe_frame_size(frames[0])
        video_path = os.path.join(video_dir, f'{vid}.yuv')
        
        with tempfile.TemporaryDirectory() as tmp:
            for idx, src_path in enumerate(frames):
                dst_path = os.path.join(tmp, f'frame_{idx:06d}.png')
                try:
                    os.symlink(os.path.abspath(src_path), dst_path)
                except Exception as e:
                    shutil.copy2(src_path, dst_path)
                    
            cmd = [
                'ffmpeg', '-y', '-framerate', f'{FPS}', '-i', os.path.join(tmp, 'frame_%06d.png'), '-pix_fmt', 'yuv420p', '-f', 'rawvideo', video_path
            ]
            subprocess.run(cmd, check=True)
            if DEBUG:
                break
    
    
    # frames = sort_frames(frame_dir)
    # width, height = probe_frame_size(frames[0])
    # with tempfile.TemporaryDirectory() as tmp:
    #     for idx, src_path in enumerate(frames):
    #         dst_name = f'frame_{idx:06d}.png'
    #         dst_path = os.path.join(tmp, dst_name)
    #         try:
    #             os.symlink(os.path.abspath(src_path), dst_path)
    #         except:
    #             import shutil
    #             shutil.copy2(src_path, dst_path)
    #     cmd = [
    #         'ffmpeg', '-y', '-framerate', f'{FPS}', '-i', f'{os.path.join(tmp, "frame_%06d.png")} -pix_fmt yuv420p -f rawvideo {video_path}'
    #     ]
    #     subprocess.run(cmd, check=True)
    
    # for frame in frames:
    #     videoID, frameID = extract_frameID(frame)
    #     video_path = os.path.join(video_dir, f'{videoID}.yuv')
    #     frame_path = os.path.join(frame_dir, frame)
    #     cmd = [
    #         'ffmpeg', '-y', '-framerate', f'{FPS}', '-i', f'{frame_path} -pix_fmt yuv420p -f rawvideo {video_path}'
    #     ]
    #     # subprocess.run(cmd, check=True)
    #     print(frame_path)
    #     print(video_path)

  0%|          | 0/2 [00:00<?, ?it/s]ffmpeg version N-112792-g7c16bf0829 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.4.0-1ubuntu1~22.04)
  configuration: --enable-gpl --enable-libx264 --enable-libx265 --enable-nonfree --enable-libvmaf --enable-version3
  libavutil      58. 32.100 / 58. 32.100
  libavcodec     60. 33.100 / 60. 33.100
  libavformat    60. 17.100 / 60. 17.100
  libavdevice    60.  4.100 / 60.  4.100
  libavfilter     9. 13.100 /  9. 13.100
  libswscale      7.  6.100 /  7.  6.100
  libswresample   4. 13.100 /  4. 13.100
  libpostproc    57.  4.100 / 57.  4.100
Input #0, image2, from '/tmp/tmp9_sypq3i/frame_%06d.png':
  Duration: 00:00:03.33, start: 0.000000, bitrate: N/A
  Stream #0:0: Video: png, rgb24(pc, gbr/unknown/unknown), 640x480, 30 fps, 30 tbr, 30 tbn
Stream mapping:
  Stream #0:0 -> #0:0 (png (native) -> rawvideo (native))
Press [q] to stop, [?] for help
Output #0, rawvideo, to '/data/ssd/zhaoy/datasets/TouchandGoDataset-v2/datas